# Notebook 2: Stage-1 Crop Classifier + LeafNet Domain Pretrain

Requires Notebook 1's output (`crop_disease/data.zip` on your Google Drive). Trains the **teacher** backbone (EfficientNetV2-S) — distillation to a mobile-sized student happens later in Notebook 4.

Flow: ImageNet weights -> finetune on LeafNet (field-domain adaptation) -> finetune on your 6-crop manifest (Stage-1: crop classifier, 6-way).

Companion docs:
- `docs/superpowers/specs/2026-08-12-crop-disease-dataset-strategy-design.md`
- `docs/superpowers/plans/2026-08-12-notebook1-ingest-unify.md`

## Task 1: Setup — install, auth, load Notebook 1's output

In [ ]:
!pip install -q timm datasets huggingface_hub pillow pandas scikit-learn

In [ ]:
import os
from google.colab import userdata, drive
from huggingface_hub import login

drive.mount('/content/drive')
login(token=userdata.get("HF_TOKEN"))
print("HF auth loaded.")

In [ ]:
import zipfile
import pandas as pd

DRIVE_ZIP = "/content/drive/MyDrive/crop_disease/data.zip"
UNIFIED_ROOT = "/content/data"

assert os.path.exists(DRIVE_ZIP), f"{DRIVE_ZIP} not found — run Notebook 1 first"
os.makedirs(UNIFIED_ROOT, exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as zf:
    zf.extractall(UNIFIED_ROOT)

manifest = pd.read_csv(os.path.join(UNIFIED_ROOT, "manifest.csv"))
print(manifest.shape)
print(manifest.groupby("crop").size())

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (Runtime -> Change runtime type -> GPU)")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Task 2: Explore LeafNet's real schema before writing loader code

In [ ]:
from datasets import load_dataset

leafnet = load_dataset("enalis/LeafNet")
print(leafnet)
print("\nFirst example keys:", list(leafnet[list(leafnet.keys())[0]][0].keys()))
print("First example (non-image fields):")
first_split = list(leafnet.keys())[0]
example = leafnet[first_split][0]
for k, v in example.items():
    if k.lower() not in ("image",):
        print(f"  {k}: {v}")

**Findings from Task 2's actual output:** LeafNet has no plain label column — only `file_name`, `image`, `caption`. Captions follow a fixed template:
- Healthy: `"a image of {Species} healthy leaves with leaves appearing normal and healthy"`
- Diseased: `"a image of {Species} leaves diseased by {Disease} with symptoms of {description...}"`

Task 3 below parses `{Species}__{Disease}` out of the caption via regex instead of reading a label column.

## Task 3: LeafNet domain-pretrain

In [ ]:
import re

LEAFNET_IMAGE_COL = "image"
LEAFNET_TRAIN_SPLIT = "train"

# Two caption forms observed (see cell-8):
#   "a image of {Species} healthy leaves with leaves appearing normal and healthy"
#   "a image of {Species} leaves diseased by {Disease} with symptoms of ..."
CAPTION_RE = re.compile(
    r"^a image of (?P<species>\w+) (?:healthy leaves with leaves appearing normal and healthy"
    r"|leaves diseased by (?P<disease>.+?) with symptoms of)"
)

def parse_caption(caption):
    m = CAPTION_RE.match(caption)
    if not m:
        return None
    species = m.group("species")
    disease = m.group("disease") or "healthy"
    return f"{species}__{disease}"

leafnet_train = leafnet[LEAFNET_TRAIN_SPLIT]
all_captions = leafnet_train["caption"]  # columnar read, doesn't touch/decode images
all_labels = [parse_caption(c) for c in all_captions]

unparsed = sum(1 for l in all_labels if l is None)
print(f"{unparsed}/{len(all_labels)} captions failed to parse")
assert unparsed / len(all_labels) < 0.05, \
    "Too many unparsed captions — caption template has more variants than cell-8's two forms, inspect the failures before continuing"

# Drop any unparsed rows rather than guessing their label.
LEAFNET_VALID_INDICES = [i for i, l in enumerate(all_labels) if l is not None]
unique_labels = sorted(set(l for l in all_labels if l is not None))
LEAFNET_LABEL_TO_IDX = {lbl: i for i, lbl in enumerate(unique_labels)}
LEAFNET_LABEL_IDS = [LEAFNET_LABEL_TO_IDX[all_labels[i]] for i in LEAFNET_VALID_INDICES]
NUM_LEAFNET_CLASSES = len(unique_labels)
print(f"{NUM_LEAFNET_CLASSES} LeafNet species__disease classes, {len(LEAFNET_VALID_INDICES)} usable images")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class LeafNetDataset(Dataset):
    def __init__(self, hf_split, indices, label_ids, image_col, transform):
        self.ds = hf_split
        self.indices = indices
        self.label_ids = label_ids
        self.image_col = image_col
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        row = self.ds[self.indices[i]]
        img = row[self.image_col].convert("RGB")
        label = self.label_ids[i]
        return self.transform(img), label

leafnet_dataset = LeafNetDataset(leafnet_train, LEAFNET_VALID_INDICES, LEAFNET_LABEL_IDS, LEAFNET_IMAGE_COL, train_tfms)
leafnet_loader = DataLoader(leafnet_dataset, batch_size=64, shuffle=True, num_workers=2, drop_last=True)
print(f"{len(leafnet_loader)} batches per epoch")

In [ ]:
import timm
import torch.nn as nn

backbone = timm.create_model("tf_efficientnetv2_s", pretrained=True, num_classes=NUM_LEAFNET_CLASSES)
backbone = backbone.to(DEVICE)
print(sum(p.numel() for p in backbone.parameters()) / 1e6, "M parameters")

In [ ]:
LEAFNET_EPOCHS = 3  # Colab free-tier GPU time is limited — start small, raise if you have Colab Pro / more quota
LEAFNET_CKPT = "/content/drive/MyDrive/crop_disease/leafnet_pretrain.pt"

optimizer = torch.optim.AdamW(backbone.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=LEAFNET_EPOCHS * len(leafnet_loader))
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

os.makedirs(os.path.dirname(LEAFNET_CKPT), exist_ok=True)

# Resume support: Colab sessions can disconnect mid-training.
start_epoch = 0
if os.path.exists(LEAFNET_CKPT):
    ckpt = torch.load(LEAFNET_CKPT, map_location=DEVICE)
    backbone.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from epoch {start_epoch}")

for epoch in range(start_epoch, LEAFNET_EPOCHS):
    backbone.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in leafnet_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = backbone(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    print(f"[LeafNet] epoch {epoch+1}/{LEAFNET_EPOCHS} loss={running_loss/total:.4f} acc={correct/total:.4f}")
    torch.save({
        "model_state": backbone.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": epoch,
    }, LEAFNET_CKPT)

print(f"LeafNet pretrain done, checkpoint at {LEAFNET_CKPT}")

## Task 4: Stage-1 crop classifier (6-way) finetune

In [ ]:
from sklearn.model_selection import train_test_split

CROPS = sorted(manifest["crop"].unique())
CROP_TO_IDX = {c: i for i, c in enumerate(CROPS)}
print("Crops:", CROP_TO_IDX)

train_df, val_df = train_test_split(
    manifest, test_size=0.15, stratify=manifest["crop"], random_state=42
)
print(f"train={len(train_df)} val={len(val_df)}")

In [ ]:
from PIL import Image

class CropDataset(Dataset):
    def __init__(self, df, crop_to_idx, transform):
        self.df = df.reset_index(drop=True)
        self.crop_to_idx = crop_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        label = self.crop_to_idx[row["crop"]]
        return self.transform(img), label

train_ds = CropDataset(train_df, CROP_TO_IDX, train_tfms)
val_ds = CropDataset(val_df, CROP_TO_IDX, eval_tfms)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
print(f"{len(train_loader)} train batches, {len(val_loader)} val batches")

In [ ]:
import numpy as np

# Crop counts are heavily imbalanced (sugarcane ~20k vs cotton ~1.7k) —
# class-weighted loss so the classifier doesn't just learn to predict the
# majority class.
class_counts = train_df["crop"].map(CROP_TO_IDX).value_counts().sort_index()
class_weights = torch.tensor((1.0 / class_counts.values) * class_counts.values.sum() / len(CROPS), dtype=torch.float32).to(DEVICE)
print("Class weights:", dict(zip(CROPS, class_weights.tolist())))

In [ ]:
# Load the LeafNet-pretrained backbone, swap its head for our 6-way crop head.
stage1_model = timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=NUM_LEAFNET_CLASSES)
leafnet_ckpt = torch.load(LEAFNET_CKPT, map_location=DEVICE)
stage1_model.load_state_dict(leafnet_ckpt["model_state"])
stage1_model.reset_classifier(num_classes=len(CROPS))
stage1_model = stage1_model.to(DEVICE)

STAGE1_EPOCHS = 8
STAGE1_CKPT = "/content/drive/MyDrive/crop_disease/stage1_crop_classifier.pt"

optimizer = torch.optim.AdamW(stage1_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE1_EPOCHS * len(train_loader))
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

start_epoch = 0
best_val_acc = 0.0
if os.path.exists(STAGE1_CKPT):
    ckpt = torch.load(STAGE1_CKPT, map_location=DEVICE)
    stage1_model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt.get("val_acc", 0.0)
    print(f"Resumed from epoch {start_epoch}, best_val_acc={best_val_acc:.4f}")

for epoch in range(start_epoch, STAGE1_EPOCHS):
    stage1_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = stage1_model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    train_loss, train_acc = running_loss / total, correct / total

    stage1_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = stage1_model(imgs)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += imgs.size(0)
    val_acc = val_correct / val_total

    print(f"[Stage1] epoch {epoch+1}/{STAGE1_EPOCHS} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

    best_val_acc = max(best_val_acc, val_acc)
    torch.save({
        "model_state": stage1_model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": epoch,
        "val_acc": val_acc,
        "crop_to_idx": CROP_TO_IDX,
    }, STAGE1_CKPT)

print(f"Stage-1 training done. Best val_acc={best_val_acc:.4f}. Checkpoint: {STAGE1_CKPT}")

## Task 5: Evaluation report + verify saved checkpoint

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

stage1_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        preds = stage1_model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CROPS))
print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(confusion_matrix(all_labels, all_preds), index=CROPS, columns=CROPS))

In [ ]:
# Reload from disk fresh (not the in-memory model) to verify the saved
# checkpoint actually works standalone — this is what Notebook 3 will load.
verify_model = timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=len(CROPS))
verify_ckpt = torch.load(STAGE1_CKPT, map_location=DEVICE)
verify_model.load_state_dict(verify_ckpt["model_state"])
verify_model = verify_model.to(DEVICE).eval()

dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    out = verify_model(dummy_input)
assert out.shape == (2, len(CROPS)), f"Expected (2, {len(CROPS)}), got {out.shape}"
print(f"Checkpoint verified: output shape {out.shape}, crop order {verify_ckpt['crop_to_idx']}")